<a href="https://colab.research.google.com/github/zemenfes-afk/Temporal-Attacks/blob/main/Temporal_intrusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1-  Dataset preprocess**

Environment Setup

In [ ]:
# Install required libraries
!pip install numpy pandas matplotlib seaborn scipy statsmodels shap scikit-learn tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.fft import fft
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf

kaggle dataset loading

In [ ]:
# 1️⃣ Install required tools
!pip install -q kagglehub pandas numpy

import kagglehub
import os
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

Download CIC-IDS2018 from Kaggle

In [ ]:
print("⬇️ Checking for CIC-IDS2018 dataset...")

raw_path = kagglehub.dataset_download("solarmainframe/ids-intrusion-csv")
print(f"✅ Dataset downloaded to: {raw_path}")

**Locate CSV Files**

In [ ]:
# List files inside dataset directory
print("📂 Available files:")
for root, dirs, files in os.walk(raw_path):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

`"Comment on the selection of the dataset"`


| Date       | Likely Content |
| ---------- | -------------- |
| 02-16-2018 | DDoS-heavy     |
| 02-20-2018 | DDoS           |
| 02-21-2018 | Mixed traffic  |
| 02-22-2018 | Web attacks    |
| 02-23-2018 | Brute force    |
| 03-01-2018 | Botnet         |
| 03-02-2018 | Infiltration   |


| Purpose        | Dataset Day    | Reason                 |
| -------------- | -------------- | ---------------------- |
| Training       | **02-21-2018** | mixed benign + attacks |
| Testing        | **02-16-2018** | heavy DDoS             |
| Generalization | **03-01-2018** | botnet                 |

This supports temporal evasion experiments.


# **Step 2 — Load ONE Dataset (Start with Mixed Traffic)**

In [ ]:
import pandas as pd

file_path = "/root/.cache/kagglehub/datasets/solarmainframe/ids-intrusion-csv/versions/1/02-21-2018.csv"

df = pd.read_csv(file_path)

print(df.shape)
df.head()

# Step 3 — Check the Columns

In [ ]:
print(df.columns)

# Step 4 — Convert Timestamp to Datetime

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S')

In [ ]:
df[['Timestamp']].head()

# Step 5 — Sort by Time (VERY IMPORTANT)

In [ ]:
df = df.sort_values(by='Timestamp')

df = df.reset_index(drop=True)

# Step 6 — Check Label Distribution

In [ ]:
df['Label'].value_counts()

# Step 7 — Convert Labels to Binary

In [ ]:
df['Attack'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

In [ ]:
df['Attack'].value_counts()

# Step 8 — Remove Problematic Values

In [ ]:
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)

df = df.dropna()

# Step 9 — Select Time-Series Features

In [ ]:
features = [
'Flow Duration',
'Tot Fwd Pkts',
'Tot Bwd Pkts',
'Flow Pkts/s',
'Flow Byts/s',
'Flow IAT Mean'
]

df_selected = df[['Timestamp'] + features + ['Attack']]

# Step 10 — Convert to Time-Series Windows

In [ ]:
df_selected = df_selected.set_index('Timestamp')

In [ ]:
time_series = df_selected.resample('1s').mean()

# Step 11 — Plot the Time Signal

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

plt.plot(time_series['Flow IAT Mean'])

plt.title("Network Traffic Time Series")
plt.show()

In [ ]:
# Calculate the total time span
total_duration = time_series.index.max() - time_series.index.min()

print(f"Start Time: {time_series.index.min()}")
print(f"End Time:   {time_series.index.max()}")
print(f"Total Span: {total_duration}")

# Step 12 — Time Domain Analysis

In [ ]:
time_series.describe()

# Step 13 — Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(time_series['Flow IAT Mean'].dropna(), lags=50)
plt.show()

# Step 14 — Frequency Domain (FFT)

In [ ]:
from scipy.fft import fft

signal = time_series['Flow IAT Mean'].dropna()

fft_vals = fft(signal)

import numpy as np

freq = np.fft.fftfreq(len(signal))

plt.plot(freq, np.abs(fft_vals))
plt.title("Frequency Spectrum")
plt.show()

# Assignment


# 3a. Descriptive Statistics

In [ ]:
import scipy.stats as stats

signal = time_series['Flow IAT Mean'].dropna()

mean = signal.mean()
variance = signal.var()
skewness = stats.skew(signal)
kurtosis = stats.kurtosis(signal)

print(mean, variance, skewness, kurtosis)

# 3b. Visualization of Temporal Patterns

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(signal)
plt.title("Network Traffic Time Series")
plt.xlabel("Time")
plt.ylabel("Flow IAT Mean")
plt.show()

# 3c. Stationarity Test
# Use ADF (Augmented Dickey-Fuller).

In [ ]:
from statsmodels.tsa.stattools import adfuller
adfuller(signal)
result = adfuller(signal)

print("ADF Statistic:", result[0])
print("p-value:", result[1])

# 3d. Autocorrelation (ACF) and Partial Autocorrelation (PACF)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plot_acf(signal)

plot_acf(signal, lags=50)
plot_pacf(signal, lags=50)

# 3e. Seasonality, Trends, Structural Breaks

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(signal, model='additive', period=60)

decomposition.plot()

### Numerical Explanation for 3e. Seasonality, Trends, Structural Breaks

In [ ]:
print("Trend Component Description:\n", decomposition.trend.describe())
print("\nSeasonal Component Description:\n", decomposition.seasonal.describe())
print("\nResidual Component Description:\n", decomposition.resid.describe())

| Statistic | Trend Component | Seasonal Component | Residual Component |
| :-------- | :-------------- | :----------------- | :----------------- |
| **Count**   | 3196            | 3256               | 3196               |
| **Mean**    | 3.92e+06        | -5.49e+02          | -2.21e+03          |
| **Std**     | 3.81e+06        | 1.04e+06           | 6.97e+06           |
| **Min**     | 2.65e+03        | -1.56e+06          | -1.56e+07          |
| **25%**     | 6.18e+03        | -8.59e+05          | -2.41e+06          |
| **50%**     | 4.10e+06        | -1.77e+05          | -4.18e+04          |
| **75%**     | 6.50e+06        | 6.65e+05           | 1.30e+06           |
| **Max**     | 1.45e+07        | 3.57e+06           | 7.92e+07           |

In [ ]:
# Extract x-axis (time) values for the seasonal component
seasonal_x_axis = decomposition.seasonal.index

# Extract y-axis (seasonal) values for the seasonal component
seasonal_y_axis = decomposition.seasonal.values

# Display the first few values and their counts
print("First 5 x-axis (Timestamp) values of Seasonal Component:\n", seasonal_x_axis[:10])
print("\nFirst 5 y-axis (Seasonal) values of Seasonal Component:\n", seasonal_y_axis[:10])
print(f"\nTotal number of x-axis values: {len(seasonal_x_axis)}")
print(f"Total number of y-axis values: {len(seasonal_y_axis)}")

In [ ]:
print("Seasonal Component (Flow IAT Mean) Values:\n", seasonal_y_axis)

### Numerical Explanation for 3f. Distribution Analysis and Outlier Detection

In [ ]:
print(f"Mean of Flow IAT Mean: {mean}")
print(f"Variance of Flow IAT Mean: {variance}")
print(f"Skewness of Flow IAT Mean: {skewness}")
print(f"Kurtosis of Flow IAT Mean: {kurtosis}")

print(f"\nQ1 (25th percentile): {Q1}")
print(f"Q3 (75th percentile): {Q3}")
print(f"IQR (Interquartile Range): {IQR}")
print(f"Number of outliers detected: {len(outliers)}")
print("First 5 detected outliers:\n", outliers.head())

# 3f. Distribution Analysis and Outlier Detection

In [ ]:
plt.hist(signal, bins=50)
plt.title("Traffic Distribution")
plt.show()

In [ ]:
Q1 = signal.quantile(0.25)
Q3 = signal.quantile(0.75)
IQR = Q3 - Q1

outliers = signal[(signal < Q1 - 1.5*IQR) | (signal > Q3 + 1.5*IQR)]

# frequency-domain analysis

In [ ]:
signal = time_series['Flow IAT Mean'].dropna()

# 4.1 Power Spectral Density (PSD)
 The PSD plot shows how the power (or energy) of your Flow IAT Mean signal is distributed across different frequencies. Peaks in the plot indicate frequencies where the signal has more energy.

In [ ]:
from scipy.signal import welch
import matplotlib.pyplot as plt
freq, psd = welch(signal.values, fs=1)

plt.figure(figsize=(10,4))
plt.semilogy(freq, psd)
plt.xlabel("Frequency")
plt.ylabel("Power Spectral Density")
plt.title("PSD of Flow IAT Mean")
plt.show()

In [ ]:
print("First 5 frequencies (Hz):\n", freq[:5])
print("First 5 power spectral density values:\n", psd[:5])
print(f"Total number of frequency bins: {len(freq)}")

# 4.2 Dominant Frequency Components
This identifies the single frequency at which the signal exhibits the highest power or energy.

In [ ]:
import numpy as np

dominant_freq = freq[np.argmax(psd)]
print("Dominant frequency:", dominant_freq)

# 4.3 Fourier Spectral Analysis (FFT)

In [ ]:
from scipy.fft import fft, fftfreq

N = len(signal)
fft_vals = fft(signal)
freqs = fftfreq(N, d=1)

plt.figure(figsize=(10,4))
plt.plot(freqs, np.abs(fft_vals))
plt.title("FFT Spectrum of Flow IAT Mean")
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.show()

# 4.5 Time-Frequency Analysis (STFT)

In [ ]:
from scipy.signal import stft

f, t, Zxx = stft(signal, fs=1)

plt.figure(figsize=(10,4))
plt.pcolormesh(t, f, np.abs(Zxx))
plt.ylabel("Frequency")
plt.xlabel("Time")
plt.title("STFT Spectrogram of Flow IAT Mean")
plt.show()

In [ ]:
print(f"Number of frequencies (f): {len(f)}")
print(f"Number of time segments (t): {len(t)}")
print(f"Shape of STFT output (Zxx): {Zxx.shape}")

# 4.6 Wavelet Multi-Resolution Analysis

In [ ]:
import pywt

coeffs = pywt.wavedec(signal, 'db4', level=4)

for i, c in enumerate(coeffs):
    print("Level", i, "Energy:", np.sum(np.square(c)))

# 4.7 Frequency-Domain Feature Extraction

In [ ]:
# Spectral Centroid: weighted average of the frequencies, with the PSD values as weights
spectral_centroid = np.sum(freq * psd) / np.sum(psd)
print(f"Spectral Centroid: {spectral_centroid}")

# Total Band Power: sum of the PSD across all frequencies
total_band_power = np.sum(psd)
print(f"Total Band Power: {total_band_power}")

# Spectral Roll-off: frequency below which 85% of the total energy is contained
# First, calculate the cumulative sum of normalized PSD
cumulative_psd = np.cumsum(psd_norm)

# Find the frequency where the cumulative sum exceeds 85%
rolloff_idx = np.where(cumulative_psd >= 0.85)[0][0]
spectral_rolloff = freq[rolloff_idx]
print(f"Spectral Roll-off (85%): {spectral_rolloff}")

# Bandwidth (using spectral standard deviation as a proxy)
# This is often used in audio processing, but can be adapted.
spectral_std = np.sqrt(np.sum(((freq - spectral_centroid)**2) * psd) / np.sum(psd))
print(f"Spectral Standard Deviation (Bandwidth proxy): {spectral_std}")

Clean the dataset